# Error comparison 

Compare **forward Euler**, **backward Euler**, and **SciPy's `solve_ivp` with RK45** on the same initial value problem.

## 1. Problem statement

We will use the reaction $A \overset{k}{\rightarrow} B$ from the earlier example:

$$\frac{dC_A}{dt} = -kC_A, \qquad C_A(0)=1\ \mathrm{mol/L}, \qquad k=0.2\ \mathrm{min}^{-1}.$$

Assuming a constant total concentration of $1\ \mathrm{mol/L}$, $C_B = 1 - C_A$.

The exact solution is $C_A(t)=C_A(0)e^{-kt}$.

In [ ]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from collections.abc import Callable
import scipy
from scipy.integrate import solve_ivp

## 2. Reuse the forward and backward Euler methods

Reuse the `forward_euler` and `backward_euler` functions defined in the previous examples.

In [ ]:
def forward_euler(func: Callable, c0: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Generic forward Euler method for initial value problem.

    Parameters
    ----------
    func : Callable
        ODE system to be solved
    c0 : np.ndarray
        Initial condition
    t : np.ndarray
        Time grid points

    Returns
    -------
    np.ndarray
        Solution of ODE system
    """
    # initialize arrays for time and solution values
    c = np.zeros((len(c0), len(t)))
    h = t[1] - t[0]

    # initial condition
    c[:, 0] = c0

    # iterate over each time step
    for i in range(1, len(t)):
        c[:, i] = c[:, i - 1] + h * func(t[i - 1], c[:, i - 1])
    return c

In [ ]:
def backward_euler(func: Callable, c0: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Generic Backward Euler method for initial value problem. Use scipy's
    fsolve to solve root finding problem.

    Parameters
    ----------
    func : Callable
        Function that defines the ODE (y' = func(t, y)).
    c0 : np.ndarray
        Initial condition.
    t : np.ndarray
        Time domain.

    Returns
    -------
    np.ndarray
        Array of solution values at the time points.
    """
    # initialize arrays for time and solution values
    c = np.zeros([len(c0), len(t)])
    dt = t[1] - t[0]

    # initial condition
    c[:, 0] = c0

    # iterate over each time step
    for i in range(len(t) - 1):
        # initial guess for y_{i+1}
        c_guess = c[:, i]

        # define backward Euler function
        def euler(c_next: np.ndarray, i: int = i) -> np.ndarray:
            return c[:, i] + dt * func(t[i + 1], c_next) - c_next

        # update solution
        c[:, i + 1] = scipy.optimize.fsolve(euler, c_guess)

    return c

## 3. Define the ODE and the analytical solution

In [ ]:
# Problem parameters
k = 0.2  # s^-1
c0 = np.array([1.0])  # mol/L
t_end = 30.0  # min

# ODE definition
def dcA(t, c):
    return -k * c

# Analytical solution
def analytical_c(t):
    return c0[0] * np.exp(-k * t)

## 3. Compare solutions

In [ ]:
# Define time grid
n = 10
t = np.linspace(0.0, t_end, n + 1)
h = t[1] - t[0]

# Forward Euler method
c_forward = forward_euler(dcA, c0, t)[0]

# Backward Euler method
c_backward = backward_euler(dcA, c0, t)[0]

# Solve_ivp method
result = solve_ivp(fun=dcA, t_span=(t[0], t[-1]), y0=c0, t_eval=t, method="RK45")
c_rk45 = result.y[0]

# Plot results
fig, ax = plt.subplots()
t_plot = np.linspace(0.0, t_end, 501)
ax.plot(t_plot, analytical_c(t_plot), "k--", label="Exact")
ax.plot(t, c_forward, "o-", label="Forward Euler")
ax.plot(t, c_backward, "s-", label="Backward Euler")
ax.plot(t, c_rk45, "^-", label="solve_ivp() RK45")
ax.set(xlabel="Time [min]", ylabel="Concentration [mol/L]",
       title=f"Solution comparison")
ax.legend()
ax.grid(alpha=0.3)
plt.show()